In [36]:
import pandas as pd
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error
import plotly.express as px

In [4]:
df_rte = pd.read_csv('../data/power_consumption_2000_2025_hourly.csv')
print(df_rte.shape)
df_rte.head()

(223486, 3)


,start_date,end_date,value
0,2000-01-01 00:00:00+01:00,2000-01-01 01:00:00+01:00,50094.0
1,2000-01-01 01:00:00+01:00,2000-01-01 02:00:00+01:00,49971.0
2,2000-01-01 02:00:00+01:00,2000-01-01 03:00:00+01:00,48666.5
3,2000-01-01 03:00:00+01:00,2000-01-01 04:00:00+01:00,45793.0
4,2000-01-01 04:00:00+01:00,2000-01-01 05:00:00+01:00,43498.5


In [24]:
df_prophet = df_rte.copy()
df_prophet.drop('start_date', axis=1, inplace=True)
df_prophet.rename(columns={'end_date': 'ds', 'value': 'y',}, inplace=True)
df_prophet['ds'] = pd.to_datetime(df_prophet['ds'], utc=True).dt.tz_convert(None)
df_prophet.head()


,ds,y
0,2000-01-01 00:00:00,50094.0
1,2000-01-01 01:00:00,49971.0
2,2000-01-01 02:00:00,48666.5
3,2000-01-01 03:00:00,45793.0
4,2000-01-01 04:00:00,43498.5


In [25]:
#train for 2 years
df_2000_02_train = df_prophet[df_prophet['ds'] < '2002-01-01']
print(df_2000_02_train.shape)
df_2000_02_train.head()

(17542, 2)


,ds,y
0,2000-01-01 00:00:00,50094.0
1,2000-01-01 01:00:00,49971.0
2,2000-01-01 02:00:00,48666.5
3,2000-01-01 03:00:00,45793.0
4,2000-01-01 04:00:00,43498.5


In [26]:
# test for 1 day
df_2000_02_test = df_prophet[(df_prophet['ds'] >= '2002-01-01') & (df_prophet['ds'] < '2002-01-02')]
print(df_2000_02_test.shape)
df_2000_02_test.head()

(24, 2)


,ds,y
17542,2002-01-01 00:00:00,63286.5
17543,2002-01-01 01:00:00,63259.5
17544,2002-01-01 02:00:00,62249.0
17545,2002-01-01 03:00:00,59413.0
17546,2002-01-01 04:00:00,57243.5


In [27]:
model = Prophet()
model.fit(df_2000_02_train)

18:29:56 - cmdstanpy - INFO - Chain [1] start processing
18:30:01 - cmdstanpy - INFO - Chain [1] done processing


In [28]:
df_pred = model.predict(df_2000_02_test)
df_pred.head(10)

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,additive_terms,additive_terms_lower,additive_terms_upper,daily,...,weekly,weekly_lower,weekly_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2002-01-01 00:00:00,48567.320986,59738.019730,67914.234844,48567.320986,48567.320986,15320.968231,15320.968231,15320.968231,-2652.959466,...,2572.201716,2572.201716,2572.201716,15401.725981,15401.725981,15401.725981,0.0,0.0,0.0,63888.289217
1,2002-01-01 01:00:00,48566.241951,58288.935439,66428.596573,48566.241951,48566.241951,13749.035466,13749.035466,13749.035466,-4270.908002,...,2617.456085,2617.456085,2617.456085,15402.487383,15402.487383,15402.487383,0.0,0.0,0.0,62315.277417
2,2002-01-01 02:00:00,48565.162916,56281.254963,64513.741013,48565.162916,48565.162916,11838.539875,11838.539875,11838.539875,-6213.426670,...,2648.678792,2648.678792,2648.678792,15403.287754,15403.287754,15403.287754,0.0,0.0,0.0,60403.702791
3,2002-01-01 03:00:00,48564.083880,55098.702683,62839.069735,48564.083880,48564.083880,10609.665020,10609.665020,10609.665020,-7461.456366,...,2666.994085,2666.994085,2666.994085,15404.127301,15404.127301,15404.127301,0.0,0.0,0.0,59173.748901
4,2002-01-01 04:00:00,48563.004845,55484.545098,63707.120889,48563.004845,48563.004845,11090.089386,11090.089386,11090.089386,-6988.511679,...,2673.594837,2673.594837,2673.594837,15405.006229,15405.006229,15405.006229,0.0,0.0,0.0,59653.094232
5,2002-01-01 05:00:00,48561.925810,57699.501076,66166.144799,48561.925810,48561.925810,13434.041214,13434.041214,13434.041214,-4641.610391,...,2669.726866,2669.726866,2669.726866,15405.924739,15405.924739,15405.924739,0.0,0.0,0.0,61995.967024
6,2002-01-01 06:00:00,48560.846775,61066.737600,69561.755099,48560.846775,48560.846775,16730.575429,16730.575429,16730.575429,-1332.980696,...,2656.673094,2656.673094,2656.673094,15406.883030,15406.883030,15406.883030,0.0,0.0,0.0,65291.422203
7,2002-01-01 07:00:00,48559.767739,64116.876839,72535.659474,48559.767739,48559.767739,19677.675791,19677.675791,19677.675791,1634.056788,...,2635.737704,2635.737704,2635.737704,15407.881300,15407.881300,15407.881300,0.0,0.0,0.0,68237.443531
8,2002-01-01 08:00:00,48558.688704,66063.037370,74259.440699,48558.688704,48558.688704,21497.483714,21497.483714,21497.483714,3480.333464,...,2608.230508,2608.230508,2608.230508,15408.919742,15408.919742,15408.919742,0.0,0.0,0.0,70056.172418
9,2002-01-01 09:00:00,48557.609669,66951.666724,74943.370538,48557.609669,48557.609669,22281.674697,22281.674697,22281.674697,4296.224452,...,2575.451695,2575.451695,2575.451695,15409.998549,15409.998549,15409.998549,0.0,0.0,0.0,70839.284366


In [68]:
fig = px.line(df_2000_02_test, x='ds', y='y', range_y=[0, None])
fig.add_trace(  px.line(df_pred, x='ds', y='yhat').data[0]  )
fig.data[1]['line']['color'] = '#4BE8E0'
fig.show()

In [41]:
# crossvalidation
df_2023_24_train = df_prophet[(df_prophet['ds'] >= '2023-01-01') & (df_prophet['ds'] < '2025-01-01')]
print(df_2023_24_train.shape)

(17542, 2)


In [39]:
m = Prophet()
m.fit(df_2023_24_train)

21:47:16 - cmdstanpy - INFO - Chain [1] start processing
21:47:25 - cmdstanpy - INFO - Chain [1] done processing


In [42]:
(17500-10000)/1000

7.5

In [43]:
df_cv = cross_validation(model, initial='10000 hours', period='1000 hours', horizon='72 hours')

  0%|          | 0/8 [00:00<?, ?it/s]21:58:51 - cmdstanpy - INFO - Chain [1] start processing
21:58:52 - cmdstanpy - INFO - Chain [1] done processing
 12%|█▎        | 1/8 [00:01<00:09,  1.41s/it]21:58:52 - cmdstanpy - INFO - Chain [1] start processing
21:58:56 - cmdstanpy - INFO - Chain [1] done processing
 25%|██▌       | 2/8 [00:04<00:15,  2.54s/it]21:58:56 - cmdstanpy - INFO - Chain [1] start processing
21:58:58 - cmdstanpy - INFO - Chain [1] done processing
 38%|███▊      | 3/8 [00:07<00:13,  2.62s/it]21:58:59 - cmdstanpy - INFO - Chain [1] start processing
21:59:01 - cmdstanpy - INFO - Chain [1] done processing
 50%|█████     | 4/8 [00:10<00:11,  2.82s/it]21:59:02 - cmdstanpy - INFO - Chain [1] start processing
21:59:04 - cmdstanpy - INFO - Chain [1] done processing
 62%|██████▎   | 5/8 [00:13<00:08,  2.73s/it]21:59:04 - cmdstanpy - INFO - Chain [1] start processing
21:59:08 - cmdstanpy - INFO - Chain [1] done processing
 75%|███████▌  | 6/8 [00:16<00:06,  3.05s/it]21:59:08 - cmds

In [44]:
df_cv

,ds,yhat,yhat_lower,yhat_upper,y,cutoff
0,2001-03-12 08:00:00,56266.923129,52583.718461,60221.082521,57969.0,2001-03-12 07:00:00
1,2001-03-12 09:00:00,57347.199611,53757.676857,61176.646815,59202.5,2001-03-12 07:00:00
2,2001-03-12 10:00:00,57909.745302,53882.645187,61604.204919,59440.0,2001-03-12 07:00:00
3,2001-03-12 11:00:00,58253.337071,54512.504722,61948.553495,59573.0,2001-03-12 07:00:00
4,2001-03-12 12:00:00,58129.685278,54555.935721,62172.980510,59331.5,2001-03-12 07:00:00
...,...,...,...,...,...,...
571,2001-12-31 19:00:00,72651.495951,68499.659001,76518.713333,65139.0,2001-12-28 23:00:00
572,2001-12-31 20:00:00,72036.149042,67822.443212,76211.082720,62596.0,2001-12-28 23:00:00
573,2001-12-31 21:00:00,70799.284077,66735.848631,74862.506507,60520.5,2001-12-28 23:00:00
574,2001-12-31 22:00:00,69670.825948,65714.476011,73905.033891,63321.0,2001-12-28 23:00:00


In [50]:
df_p = performance_metrics(df_cv, rolling_window=0)
df_p.head(10)

,horizon,mse,rmse,mae,mape,mdape,smape,coverage
0,0 days 01:00:00,2.199676e+07,4690.069876,4179.171907,0.092598,0.089128,0.090554,0.500
1,0 days 02:00:00,1.761923e+07,4197.526664,3646.367948,0.081039,0.086000,0.080235,0.500
2,0 days 03:00:00,1.529260e+07,3910.574912,3526.864305,0.079451,0.070023,0.078976,0.750
3,0 days 04:00:00,1.836263e+07,4285.164235,3524.164196,0.080472,0.076283,0.079772,0.500
4,0 days 05:00:00,2.411381e+07,4910.581237,3899.050078,0.088413,0.075313,0.087548,0.625
5,0 days 06:00:00,3.095125e+07,5563.384984,4005.754528,0.084493,0.048977,0.082486,0.750
6,0 days 07:00:00,4.009803e+07,6332.300664,4543.666045,0.093844,0.047666,0.090545,0.750
7,0 days 08:00:00,4.053408e+07,6366.638407,4230.913977,0.084876,0.047775,0.081070,0.750
8,0 days 09:00:00,3.537739e+07,5947.889918,3837.929622,0.075933,0.034107,0.072475,0.750
9,0 days 10:00:00,2.719458e+07,5214.842164,3444.845059,0.066901,0.035597,0.064318,0.750


In [54]:
# croos_validation cutoff fixé
cutoffs = pd.to_datetime([
    '2024-02-05 04:00:00', 
    '2024-04-10 07:00:00',
    '2024-06-15 11:00:00',
    '2024-08-20 15:00:00',
    '2024-10-25 20:00:00',
    '2024-12-27 23:00:00'])

In [55]:
df_cv2 = cross_validation(m, cutoffs=cutoffs, horizon="72 hours", parallel="processes")

22:42:53 - cmdstanpy - INFO - Chain [1] start processing
22:42:53 - cmdstanpy - INFO - Chain [1] start processing
22:42:53 - cmdstanpy - INFO - Chain [1] start processing
22:42:53 - cmdstanpy - INFO - Chain [1] start processing
22:42:53 - cmdstanpy - INFO - Chain [1] start processing
22:42:53 - cmdstanpy - INFO - Chain [1] start processing
22:43:01 - cmdstanpy - INFO - Chain [1] done processing
22:43:01 - cmdstanpy - INFO - Chain [1] done processing
22:43:07 - cmdstanpy - INFO - Chain [1] done processing
22:43:09 - cmdstanpy - INFO - Chain [1] done processing
22:43:09 - cmdstanpy - INFO - Chain [1] done processing
22:43:09 - cmdstanpy - INFO - Chain [1] done processing


In [56]:
df_p = performance_metrics(df_cv2, rolling_window=0)
df_p.head(24)

,horizon,mse,rmse,mae,mape,mdape,smape,coverage
0,0 days 01:00:00,1.488937e+07,3858.674043,2977.159159,0.053420,0.051971,0.055653,0.666667
1,0 days 02:00:00,2.075865e+07,4556.165625,3121.277082,0.053641,0.031078,0.056699,0.666667
2,0 days 03:00:00,2.966424e+07,5446.489222,3971.937587,0.069695,0.046982,0.073265,0.666667
3,0 days 04:00:00,2.839021e+07,5328.246413,4259.381766,0.078501,0.074296,0.081046,0.666667
4,0 days 05:00:00,2.200137e+07,4690.562131,3527.834753,0.065729,0.068751,0.067511,0.666667
5,0 days 06:00:00,1.589054e+07,3986.294106,2838.868196,0.053173,0.044968,0.054430,0.833333
6,0 days 07:00:00,1.511361e+07,3887.623188,2580.888154,0.047908,0.034773,0.048902,0.833333
7,0 days 08:00:00,1.480088e+07,3847.191234,3083.657877,0.062519,0.060135,0.062613,0.833333
8,0 days 09:00:00,1.591046e+07,3988.791421,3365.353250,0.068890,0.072552,0.068600,0.833333
9,0 days 10:00:00,2.215432e+07,4706.837727,4046.742818,0.081979,0.080395,0.082660,0.500000


## durée d'entrainement 3 ans

In [ ]:
df_2022_24_train = df_prophet[(df_prophet['ds'] >= '2022-01-01') & (df_prophet['ds'] < '2025-01-01')]
m3y = Prophet()
m3y.fit(df_2022_24_train)

22:48:54 - cmdstanpy - INFO - Chain [1] start processing
22:49:06 - cmdstanpy - INFO - Chain [1] done processing


In [59]:
df_cv3 = cross_validation(m3y, cutoffs=cutoffs, horizon="72 hours", parallel="processes")

22:50:28 - cmdstanpy - INFO - Chain [1] start processing
22:50:28 - cmdstanpy - INFO - Chain [1] start processing
22:50:28 - cmdstanpy - INFO - Chain [1] start processing
22:50:28 - cmdstanpy - INFO - Chain [1] start processing
22:50:28 - cmdstanpy - INFO - Chain [1] start processing
22:50:28 - cmdstanpy - INFO - Chain [1] start processing
22:50:46 - cmdstanpy - INFO - Chain [1] done processing
22:50:48 - cmdstanpy - INFO - Chain [1] done processing
22:50:50 - cmdstanpy - INFO - Chain [1] done processing
22:50:51 - cmdstanpy - INFO - Chain [1] done processing
22:50:54 - cmdstanpy - INFO - Chain [1] done processing
22:50:55 - cmdstanpy - INFO - Chain [1] done processing


In [60]:
df_p3 = performance_metrics(df_cv3, rolling_window=0)
df_p3.head(24)

,horizon,mse,rmse,mae,mape,mdape,smape,coverage
0,0 days 01:00:00,1.961800e+07,4429.221467,3187.006440,0.056550,0.044748,0.059314,0.833333
1,0 days 02:00:00,2.432405e+07,4931.941436,3349.814155,0.058232,0.026928,0.061821,0.666667
2,0 days 03:00:00,3.299839e+07,5744.422148,4053.015172,0.071089,0.043164,0.075428,0.666667
3,0 days 04:00:00,3.153273e+07,5615.401244,4297.122219,0.079184,0.076196,0.082496,0.666667
4,0 days 05:00:00,2.432576e+07,4932.114664,3722.765068,0.069714,0.069970,0.072141,0.666667
5,0 days 06:00:00,1.692519e+07,4114.022980,3033.263795,0.056515,0.050738,0.058192,0.666667
6,0 days 07:00:00,1.485496e+07,3854.213015,2964.258343,0.055859,0.053668,0.057025,0.666667
7,0 days 08:00:00,1.554347e+07,3942.520302,3516.313260,0.071289,0.085435,0.071363,0.666667
8,0 days 09:00:00,1.794771e+07,4236.473292,3803.959980,0.077464,0.091753,0.077226,0.666667
9,0 days 10:00:00,2.437219e+07,4936.819707,4303.628239,0.085617,0.097365,0.086368,0.500000


### Durée d'entrainement 4 ans

In [61]:
df_2021_24_train = df_prophet[(df_prophet['ds'] >= '2021-01-01') & (df_prophet['ds'] < '2025-01-01')]
m4y = Prophet()
m4y.fit(df_2021_24_train)

22:54:37 - cmdstanpy - INFO - Chain [1] start processing
22:54:47 - cmdstanpy - INFO - Chain [1] done processing


In [62]:
df_cv4 = cross_validation(m4y, cutoffs=cutoffs, horizon="72 hours", parallel="processes")

22:55:19 - cmdstanpy - INFO - Chain [1] start processing
22:55:19 - cmdstanpy - INFO - Chain [1] start processing
22:55:19 - cmdstanpy - INFO - Chain [1] start processing
22:55:19 - cmdstanpy - INFO - Chain [1] start processing
22:55:19 - cmdstanpy - INFO - Chain [1] start processing
22:55:19 - cmdstanpy - INFO - Chain [1] start processing
22:55:35 - cmdstanpy - INFO - Chain [1] done processing
22:55:35 - cmdstanpy - INFO - Chain [1] done processing
22:55:38 - cmdstanpy - INFO - Chain [1] done processing
22:55:40 - cmdstanpy - INFO - Chain [1] done processing
22:55:41 - cmdstanpy - INFO - Chain [1] done processing
22:55:43 - cmdstanpy - INFO - Chain [1] done processing


In [63]:
df_p4 = performance_metrics(df_cv4, rolling_window=0)
df_p4.head(24)

,horizon,mse,rmse,mae,mape,mdape,smape,coverage
0,0 days 01:00:00,1.698440e+07,4121.213570,2753.147326,0.048597,0.029513,0.050765,0.833333
1,0 days 02:00:00,1.649842e+07,4061.824244,2412.367511,0.042637,0.025536,0.045050,0.833333
2,0 days 03:00:00,2.166495e+07,4654.562184,3118.020663,0.056603,0.039770,0.059527,0.833333
3,0 days 04:00:00,2.114674e+07,4598.558401,3358.757439,0.064871,0.057867,0.066960,0.833333
4,0 days 05:00:00,1.518688e+07,3897.034741,2805.838409,0.055788,0.048493,0.057183,0.833333
5,0 days 06:00:00,9.538997e+06,3088.526602,2050.835106,0.041069,0.020282,0.041962,0.833333
6,0 days 07:00:00,7.188654e+06,2681.166604,1975.732027,0.040539,0.024934,0.040911,0.833333
7,0 days 08:00:00,9.562517e+06,3092.332010,2529.378366,0.055865,0.065540,0.055274,0.833333
8,0 days 09:00:00,1.271826e+07,3566.267616,2884.677434,0.062972,0.067762,0.062111,0.833333
9,0 days 10:00:00,1.741967e+07,4173.687314,3356.066700,0.070431,0.078581,0.070295,0.833333


### Durée d'entrainement 18 mois

In [64]:
df_20235_24_train = df_prophet[(df_prophet['ds'] >= '2023-07-01') & (df_prophet['ds'] < '2025-01-01')]
m18m = Prophet()
m18m.fit(df_20235_24_train)

22:59:36 - cmdstanpy - INFO - Chain [1] start processing
22:59:39 - cmdstanpy - INFO - Chain [1] done processing


In [65]:
df_cv5 = cross_validation(m18m, cutoffs=cutoffs, horizon="72 hours", parallel="processes")

23:00:00 - cmdstanpy - INFO - Chain [1] start processing
23:00:00 - cmdstanpy - INFO - Chain [1] start processing
23:00:00 - cmdstanpy - INFO - Chain [1] start processing
23:00:00 - cmdstanpy - INFO - Chain [1] start processing
23:00:00 - cmdstanpy - INFO - Chain [1] start processing
23:00:00 - cmdstanpy - INFO - Chain [1] start processing
23:00:00 - cmdstanpy - INFO - Chain [1] done processing
23:00:01 - cmdstanpy - INFO - Chain [1] done processing
23:00:01 - cmdstanpy - INFO - Chain [1] done processing
23:00:02 - cmdstanpy - INFO - Chain [1] done processing
23:00:02 - cmdstanpy - INFO - Chain [1] done processing
23:00:03 - cmdstanpy - INFO - Chain [1] done processing


In [66]:
df_p5 = performance_metrics(df_cv5, rolling_window=0)
df_p5.head(24)

,horizon,mse,rmse,mae,mape,mdape,smape,coverage
0,0 days 01:00:00,1.059816e+07,3255.481312,2561.605206,0.048182,0.031379,0.047738,0.666667
1,0 days 02:00:00,4.385753e+06,2094.219011,1928.706283,0.036316,0.036884,0.036534,1.000000
2,0 days 03:00:00,3.478468e+06,1865.065096,1598.896005,0.031184,0.028652,0.031578,1.000000
3,0 days 04:00:00,4.566574e+06,2136.954365,1961.857194,0.040259,0.037792,0.040142,1.000000
4,0 days 05:00:00,3.674921e+06,1917.008359,1558.283402,0.031661,0.032217,0.031313,1.000000
5,0 days 06:00:00,4.395535e+06,2096.553201,1497.073055,0.028502,0.022312,0.028273,1.000000
6,0 days 07:00:00,4.501776e+06,2121.738911,1602.960659,0.029794,0.027373,0.029345,1.000000
7,0 days 08:00:00,7.728374e+06,2779.995246,2070.737646,0.040596,0.025419,0.039282,1.000000
8,0 days 09:00:00,9.730475e+06,3119.370953,2479.457758,0.051694,0.043334,0.049948,1.000000
9,0 days 10:00:00,1.107142e+07,3327.374319,2865.215867,0.062375,0.067923,0.062036,0.833333
